In [1]:
import sys
sys.path.insert(0, "/D:\\Snowpole Detection\\ultralytics4channel-0a38736761a770f7f7dd80064e20b2d9624eda5b")  # parent of the ultralytics package

import ultralytics
from ultralytics import YOLO

print("Ultralytics module file:", ultralytics.__file__)


Ultralytics module file: d:\Snowpole Detection\cuda-env\Lib\site-packages\ultralytics\__init__.py


In [2]:
import cv2
import shutil
import yaml
import numpy as np
from pathlib import Path

ROOT = Path("SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset")

RANGE_ROOT  = ROOT / "range"
LABELS_ROOT = ROOT / "labels"
OUT_ROOT    = ROOT / "range_only"

(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

def preprocess_range(img):
    img = cv2.resize(img, (1024, 1024))
    img = cv2.equalizeHist(img)
    img = img.astype(np.float32) / 255.0
    return img

def make_split(split):
    img_out = OUT_ROOT / "images" / split
    lbl_out = OUT_ROOT / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for f in (LABELS_ROOT / split).glob("*.txt"):
        shutil.copy2(f, lbl_out / f.name)

    for p in (RANGE_ROOT / split).glob("*.*"):
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = preprocess_range(img)
        out = (img * 255).astype(np.uint8)
        out = np.repeat(out[:, :, None], 4, axis=2)
        cv2.imwrite(str(img_out / f"{p.stem}.png"), out)

for s in ["train", "valid", "test"]:
    make_split(s)

data_yaml = OUT_ROOT / "data.yaml"
cfg = {
    "path": str(OUT_ROOT),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": 1,
    "names": ["snow_pole"],
    "channels": 1
}
with open(data_yaml, "w") as f:
    yaml.safe_dump(cfg, f)


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.yaml")

model.train(
    data="SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\range_only\\data.yaml",
    imgsz=1024,
    epochs=300,
    patience=40,
    batch=4,
    device=0,
    project="Ablation_Range",
    name="range_only",
    amp=False,
    augment=False,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    auto_augment=None,
    erasing=0.0,
    workers=0,
)


New https://pypi.org/project/ultralytics/8.3.246 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.5  Python-3.11.9 torch-2.5.1+cu121 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: -1
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


: 